In [14]:
import pandas as pd

BASE = "/home/ubuntu/string-matching-algorithm/"

def load(name):
    try:
        return pd.read_csv(BASE + name, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(BASE + name, encoding="latin-1")

imdb_tv = load("imdb_tv.csv")
raw_names = load("raw_names.csv")

for name, df in [("IMDB", imdb_tv), ("RAW INPUT", raw_names)]:
    print(f"\n=== {name} ===  rows, cols: {df.shape}")
    print(df.dtypes.to_string())
    print("empty values:", df.isna().sum().to_dict())
    display(df.head(10))


=== IMDB ===  rows, cols: (284555, 1)
TitleUS    str
empty values: {'TitleUS': 0}


,TitleUS
0,#Hashtag: The Series
1,#IMomSoHard
2,...i Hetluften
3,099 Central
4,12 Eylül 1980 Darbe
5,12 TeEn
6,13 Reflections
7,16 for '16: The Contenders
8,1992: Berlusconi Rising
9,2000 Not Out



=== RAW INPUT ===  rows, cols: (174021, 1)
nielsen_program_name    str
empty values: {'nielsen_program_name': 0}


,nielsen_program_name
0,INFOMERCIAL
1,LOVE IT OR LIST IT
2,QUE NO TE CUENTEN!
3,LOST LAND OF THE TIGER
4,FAIRLY ODDPARENTS
5,CATDOG
6,SIGNED SEALED DELIVERED
7,DOKI
8,DAILY SHOW
9,ABSOLUTE HIP HOP


In [15]:
imdb_tv

,TitleUS
0,#Hashtag: The Series
1,#IMomSoHard
2,...i Hetluften
3,099 Central
4,12 Eylül 1980 Darbe
...,...
284550,¿Quién es quién?
284551,¿Seguro que es seguro?
284552,Årsrevyen
284553,É Urgente o Amor


In [ ]:
import unicodedata, random
from collections import Counter

# ---------------------------------------------------------------------------
# STEP 1: TOKENIZER
# Knows only what Unicode says about each character (letter, digit, mark, other).
# No word lists, no patterns, nothing learned from one particular file.
# ---------------------------------------------------------------------------

LETTERS_WITHOUT_ACCENT_FORM = str.maketrans({
    "ß": "ss", "æ": "ae", "Æ": "AE", "œ": "oe", "Œ": "OE", "ø": "o", "Ø": "O",
    "ł": "l", "Ł": "L", "đ": "d", "Đ": "D", "ð": "d", "Ð": "D", "þ": "th", "Þ": "TH", "ı": "i",
})

def normalize(text):
    """Capitals, accents removed, runs of spaces made one space. Nothing else changes."""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not ("\u0300" <= ch <= "\u036f"))
    text = unicodedata.normalize("NFKC", text).translate(LETTERS_WITHOUT_ACCENT_FORM).upper()
    return " ".join(text.split())

def char_class(ch):
    kind = unicodedata.category(ch)[0]
    return "letters" if kind == "L" else "digits" if kind == "N" else "mark" if kind == "M" else "other"

def split_units(text):
    """Cuts a name into units.
    A unit = letters and digits written directly against each other (8P, 30P, H2O, POTTER23).
    Returns (units, tail):
      units = list of (sep, unit, shape)
        sep   = exactly what was written between the previous unit and this one (' ', '/', ': ', '' at the start)
        shape = every digit becomes 9, every run of letters becomes A   (830P -> 999A, 4202 -> 9999, SUN -> A)
      tail  = whatever comes after the last unit (e.g. the final '.' of S.W.A.T.)
    Nothing is dropped: sep + unit for every unit, then tail, gives back the normalized name exactly."""
    units, sep, cur, shape, last, start = [], "", "", "", None, ""
    for ch in normalize(text):
        c = char_class(ch)
        if c == "mark" and cur:                 # marks (e.g. Hindi vowel signs) stay inside their word
            cur += ch
            continue
        if c in ("letters", "digits"):
            if not cur:                         # a new unit starts; remember what stood before it
                start, sep = sep, ""
            cur += ch
            if c == "digits":
                shape += "9"
            elif last != "letters":
                shape += "A"
            last = c
            continue
        if cur:                                 # space, punctuation or symbol ends the unit
            units.append((start, cur, shape))
            cur, shape, last = "", "", None
        sep += ch
    if cur:
        units.append((start, cur, shape))
    return units, sep

def rebuild(units, tail):
    return "".join(sep + unit for sep, unit, shape in units) + tail

def is_tight(sep):
    """Written together, no space: SPIDER-MAN, GREY'S, M*A*S*H, 20/20."""
    return sep != "" and " " not in sep

def is_clear_break(sep):
    """Punctuation AND a space: 'CSI: MIAMI', 'STAR TREK - VOYAGER', 'SIGNED, SEALED'."""
    return " " in sep and sep.strip() != ""

def unit_joins(units):
    """Neighbouring units stuck together where there is no clear break, so ODD PARENTS can meet ODDPARENTS,
    GREY'S can meet GREYS and M*A*S*H can meet MASH. Used only for look-ups; the units never change."""
    joins = []
    for (s1, u1, _), (s2, u2, _) in zip(units, units[1:]):
        if not is_clear_break(s2):
            joins.append(u1 + u2)
    run = []                                    # 3 or more units written tightly together -> one join (MASH, 911)
    for i, (sep, unit, _) in enumerate(units):
        if i > 0 and is_tight(sep):
            run.append(unit)
        else:
            if len(run) >= 3:
                joins.append("".join(run))
            run = [unit]
    if len(run) >= 3:
        joins.append("".join(run))
    return joins

# ---- check 1: random text from all of Unicode is always rebuilt exactly ----
random.seed(0)
def random_text():
    return "".join(chr(random.choice([random.randrange(32, 127), random.randrange(160, 0x3000), random.randrange(0x3000, 0xD7FF),
                                      random.randrange(0xE000, 0xFFFF), random.randrange(0x10000, 0x1FFFF)]))
                   if random.random() > 0.15 else " " for _ in range(random.randrange(0, 40)))
tests = [random_text() for _ in range(20000)]
assert all(rebuild(*split_units(t)) == normalize(t) for t in tests)
print("Check 1 passed: 20,000 random strings from all of Unicode rebuilt exactly.\n")

# ---- check 2: examples ----
for t in ["WHOSE LINE SAT 8:30P", "HARRY POTTER23", "20/20 SATURDAY-3/24", "FAIRLY ODD PARENTS", "Grey's Anatomy",
          "M*A*S*H 6P", "H2O: Just Add Water", "S.W.A.T.", "Шерлок Холмс №5"]:
    units, tail = split_units(t)
    print(f"{t:22} units={[u for s, u, h in units]}  shapes={[h for s, u, h in units]}")
    print(f"{'':22} seps={[s for s, u, h in units]}  tail={tail!r}  joins={unit_joins(units)}")

Check 1 passed: 20,000 random strings from all of Unicode rebuilt exactly.

WHOSE LINE SAT 8:30P   units=['WHOSE', 'LINE', 'SAT', '8', '30P']  shapes=['A', 'A', 'A', '9', '99A']
                       seps=['', ' ', ' ', ' ', ':']  tail=''  joins=['WHOSELINE', 'LINESAT', 'SAT8', '830P']
HARRY POTTER23         units=['HARRY', 'POTTER23']  shapes=['A', 'A99']
                       seps=['', ' ']  tail=''  joins=['HARRYPOTTER23']
20/20 SATURDAY-3/24    units=['20', '20', 'SATURDAY', '3', '24']  shapes=['99', '99', 'A', '9', '99']
                       seps=['', '/', ' ', '-', '/']  tail=''  joins=['2020', '20SATURDAY', 'SATURDAY3', '324', 'SATURDAY324']
FAIRLY ODD PARENTS     units=['FAIRLY', 'ODD', 'PARENTS']  shapes=['A', 'A', 'A']
                       seps=['', ' ', ' ']  tail=''  joins=['FAIRLYODD', 'ODDPARENTS']
Grey's Anatomy         units=['GREY', 'S', 'ANATOMY']  shapes=['A', 'A', 'A']
                       seps=['', "'", ' ']  tail=''  joins=['GREYS', 'SANATOMY']
M*A*S*H 6P 

In [16]:
import pandas as pd
pd.set_option("display.max_colwidth", 70)

# Start clean: keep only the name column (drops the columns made by the earlier experiments)
imdb_tv   = imdb_tv[["TitleUS"]].copy()
raw_names = raw_names[["nielsen_program_name"]].copy()

# Tokenise both files with the SAME tokenizer
for df, col in [(imdb_tv, "TitleUS"), (raw_names, "nielsen_program_name")]:
    result = [split_units(s) for s in df[col]]
    df["units"] = [u for u, tail in result]                          # list of (sep, unit, shape)
    df["tail"]  = [tail for u, tail in result]
    df["joins"] = [unit_joins(u) for u in df["units"]]
    rebuilt_ok = all(rebuild(u, t) == normalize(s) for u, t, s in zip(df["units"], df["tail"], df[col]))
    print(f"{col}: {len(df):,} names, every name rebuilt exactly: {rebuilt_ok}")

# Short summary
for name, df in [("IMDb", imdb_tv), ("Nielsen", raw_names)]:
    units  = [u for us in df["units"] for s, u, h in us]
    shapes = Counter(h for us in df["units"] for s, u, h in us)
    print(f"\n{name}: {len(units):,} units, {len(set(units)):,} different units, {len(shapes):,} different shapes")
    print("  most common shapes:", ", ".join(f"{h}: {n:,}" for h, n in shapes.most_common(12)))

# Look at a few names: each unit, its shape, and the separator written before it
sample = raw_names.sample(12, random_state=3)
display(pd.DataFrame({
    "name":       sample["nielsen_program_name"],
    "units":      sample["units"].map(lambda us: " | ".join(u for s, u, h in us)),
    "shapes":     sample["units"].map(lambda us: " | ".join(h for s, u, h in us)),
    "separators": sample["units"].map(lambda us: [s for s, u, h in us][1:]),
}))

TitleUS: 284,555 names, every name rebuilt exactly: True
nielsen_program_name: 174,021 names, every name rebuilt exactly: True

IMDb: 875,278 units, 143,458 different units, 38 different shapes
  most common shapes: A: 859,617, 9: 4,913, 99: 4,007, 9999: 2,478, 999: 1,294, A9: 782, 9A: 587, 99A: 485, A99: 357, A9A: 327, A999: 114, 999A: 48

Nielsen: 716,956 units, 33,891 different units, 34 different shapes
  most common shapes: A: 538,528, 99: 49,975, 9: 49,791, 9A: 31,129, 99A: 17,454, 999A: 8,153, 9999: 6,664, A9: 5,459, 9999A: 3,450, 999: 2,550, A99: 1,524, A9A: 572


,name,units,shapes,separators
67520,DPLUS FL: PARA CAUGHT CAM,DPLUS | FL | PARA | CAUGHT | CAM,A | A | A | A | A,"[ , : , , ]"
125838,SELECCION NAT?L MEXICANA,SELECCION | NAT | L | MEXICANA,A | A | A | A,"[ , ?, ]"
162258,PAPER MOON,PAPER | MOON,A | A,[ ]
91452,NATURE GONE WILD 5A,NATURE | GONE | WILD | 5A,A | A | A | 9A,"[ , , ]"
153248,2024 VP DEBATES 10 01,2024 | VP | DEBATES | 10 | 01,9999 | A | A | 99 | 99,"[ , , , ]"
1519,UFC TONIGHT,UFC | TONIGHT,A | A,[ ]
161359,PARANORMAL STATE SU 6A,PARANORMAL | STATE | SU | 6A,A | A | A | 9A,"[ , , ]"
38009,FOSTERS:DESTINATION IMAGN,FOSTERS | DESTINATION | IMAGN,A | A | A,"[:, ]"
121031,"REAL, THE SUN 5A",REAL | THE | SUN | 5A,A | A | A | 9A,"[, , , ]"
2768,BLESS THIS MESS,BLESS | THIS | MESS,A | A | A,"[ , ]"


In [21]:
from collections import defaultdict, Counter

# ===========================================================================
# STEP 2: split glued units, count how Nielsen uses each unit, give each unit a code score
# ===========================================================================

# ---- settings (all tuned later in Stage 4) ----
K_SPREAD       = 100   # a unit seen after 100 different programs gets half marks for spread
K_LIFT         = 5     # a unit 5x more common in Nielsen than in IMDb gets half marks for lift
M_BLEND        = 200   # numbers lean on their shape until seen in a few hundred names
SPLIT_MAX_SEEN = 20    # a glued unit seen this often stays whole (SUB17, U20)
SPLIT_MIN_WORD = 3     # the letter part must be 3+ letters and appear alone in 3+ IMDb titles or Nielsen names
PAIR_MIN_SEEN  = 10    # a written pair (20/20) needs this many sightings before it can protect
PAIR_FIXED_MAX = 0.25  # a pair this fixed is a name (20/20: 5 different units before it in 76 names)
PAIR_NAME_MAX  = 0.15  # ...and its own code score must be low (M-F is fixed but follows 553 programs)
PAIR_MIN_CODE  = 0.5   # pairs only matter when both units look like codes alone (20, 20) - not SIMPSONS-SP
PAIR_MIN_IMDB  = 2     # ...and IMDb must write the same pair in 2+ titles (20/20, 24/7, 9-1-1), so 7/31 never protects

N_IMDB = len(imdb_tv)

def shape_of(unit):
    out, last = "", None
    for ch in unit:
        c = char_class(ch)
        if c == "digits":
            out += "9"
        elif c == "letters" and last != "letters":
            out += "A"
        if c != "mark":
            last = c
    return out

def written_pairs(units):
    """Neighbouring units written with no space between them: (index of 2nd unit, '20/20', '99/99')."""
    return [(i, units[i-1][1] + units[i][0] + units[i][1], units[i-1][2] + units[i][0] + units[i][2])
            for i in range(1, len(units)) if is_tight(units[i][0])]

# ---- what IMDb titles contain (the "how common is this in real titles" baseline) ----
imdb_units  = Counter(u for us in imdb_tv["units"] for u in {u for s, u, h in us})
imdb_joins  = Counter(j for js in imdb_tv["joins"] for j in set(js))
imdb_shapes = Counter(h for us in imdb_tv["units"] for h in {h for s, u, h in us})
imdb_pairs  = Counter(p for us in imdb_tv["units"] for p in {p for i, p, ph in written_pairs(us)})

# ---- 2a. split glued units such as POTTER23 -> POTTER + 23 ----
nielsen_seen = Counter(u for us in raw_names["units"] for u in {u for s, u, h in us})

def leading_letters(u):
    k = 0
    while k < len(u) and char_class(u[k]) in ("letters", "mark"):
        k += 1
    return u[:k]

def trailing_letters(u):
    k = len(u)
    while k > 0 and char_class(u[k-1]) in ("letters", "mark"):
        k -= 1
    return u[k:]

def is_known_word(w):
    return len(w) >= SPLIT_MIN_WORD and (imdb_units[w] >= SPLIT_MIN_WORD or nielsen_seen[w] >= SPLIT_MIN_WORD)

def split_glued(units):
    out = []
    for sep, u, h in units:
        glued = "A" in h and "9" in h
        if glued and u not in imdb_units and u not in imdb_joins and nielsen_seen[u] < SPLIT_MAX_SEEN:
            head, tail = leading_letters(u), trailing_letters(u)
            if head and head != u and is_known_word(head):              # POTTER23, SAT1030A
                out += [(sep, head, "A"), ("", u[len(head):], shape_of(u[len(head):]))]
                continue
            if tail and tail != u and is_known_word(tail):              # 2NIGHT
                out += [(sep, u[:-len(tail)], shape_of(u[:-len(tail)])), ("", tail, "A")]
                continue
        out.append((sep, u, h))
    return out

before = raw_names["units"].map(len).sum()
raw_names["units"] = raw_names["units"].map(split_glued)
assert all(rebuild(u, t) == normalize(s) for u, t, s in zip(raw_names["units"], raw_names["tail"], raw_names["nielsen_program_name"]))
print(f"Glued units split: {raw_names['units'].map(len).sum() - before:,} (every name still rebuilds exactly)")

# ---- 2b. Nielsen spread counts (for units, shapes and written pairs) ----
def new_store():
    return {"names": Counter(), "prev": defaultdict(set), "first": defaultdict(set)}

stores = {"unit": new_store(), "shape": new_store(), "pair": new_store()}
n_nielsen = 0

def add_name(units):
    """Adds one Nielsen name to the running counts. Call again for every new batch; counts are never reset."""
    global n_nielsen
    n_nielsen += 1
    if not units:
        return
    first = units[0][1]
    items = []                                  # (store, key, unit before it, first unit of the name)
    for i, (sep, u, h) in enumerate(units):
        prev, ctx = (units[i-1][1], first) if i else ("^", "^")
        items += [("unit", u, prev, ctx), ("shape", h, prev, ctx)]
    for i, p, ph in written_pairs(units):
        prev, ctx = (units[i-2][1], first) if i >= 2 else ("^", "^")
        items.append(("pair", p, prev, ctx))
    for kind, key, prev, ctx in items:
        stores[kind]["prev"][key].add(prev)
        stores[kind]["first"][key].add(ctx)
    for kind, key in {(kind, key) for kind, key, prev, ctx in items}:
        stores[kind]["names"][key] += 1

for us in raw_names["units"]:
    add_name(us)

# ---- 2c. code score ----
def raw_score(kind, key, imdb_count):
    """0..1: high when the item follows many different programs AND is far more common in Nielsen than in IMDb."""
    st = stores[kind]
    n = st["names"][key]
    if n == 0:
        return 0.0, 0
    spread = max(len(st["prev"][key]), len(st["first"][key]))
    lift = (n / n_nielsen) / ((imdb_count + 1) / N_IMDB)
    return (spread / (spread + K_SPREAD)) * (lift / (lift + K_LIFT)), n

score_cache = {}
def unit_score(u, h):
    if (u, h) not in score_cache:
        s_exact, n = raw_score("unit", u, imdb_units[u])
        if "9" in h:                            # numbers: lean on the shape until the exact text is seen often
            s_shape, _ = raw_score("shape", h, imdb_shapes[h])
            w = n / (n + M_BLEND)
            s_exact = w * s_exact + (1 - w) * s_shape
        score_cache[(u, h)] = s_exact
    return score_cache[(u, h)]

def code_scores(units):
    scores = [unit_score(u, h) for s, u, h in units]
    for i, p, ph in written_pairs(units):       # 20/20, 24/7: a pair that behaves like a name protects both units
        st = stores["pair"]
        n = st["names"][p]
        spread = max(len(st["prev"][p]), len(st["first"][p]))
        if (scores[i-1] < PAIR_MIN_CODE or scores[i] < PAIR_MIN_CODE or imdb_pairs[p] < PAIR_MIN_IMDB
                or n < PAIR_MIN_SEEN or spread / n > PAIR_FIXED_MAX):
            continue
        ps, _ = raw_score("pair", p, imdb_pairs[p])
        if ps < PAIR_NAME_MAX:                  # always the same company and rare after other programs -> a name
            scores[i-1], scores[i] = min(scores[i-1], ps), min(scores[i], ps)
    return scores

raw_names["code"] = raw_names["units"].map(code_scores)

# ---- checks ----
frequent = [u for u, n in stores["unit"]["names"].items() if n >= 20]
ranked = sorted(frequent, key=lambda u: unit_score(u, shape_of(u)))
fmt = lambda us: ", ".join(f"{u} {unit_score(u, shape_of(u)):.2f}" for u in us)
print(f"\nUnits seen in 20+ names: {len(frequent):,}")
print("\n100 MOST code-like:\n" + fmt(ranked[::-1][:100]))
print("\n100 LEAST code-like:\n" + fmt(ranked[:100]))
middle = [u for u in ranked if 0.3 <= unit_score(u, shape_of(u)) <= 0.6]
print(f"\nMiddle zone 0.3-0.6 ({len(middle):,} units), most frequent 60:")
print(fmt(sorted(middle, key=lambda u: -stores['unit']['names'][u])[:60]))

def show(name):
    us = split_glued(split_units(name)[0])
    return "  ".join(f"{u} {c:.2f}" for (s, u, h), c in zip(us, code_scores(us)))

print("\nExamples (unit and its code score):")
for t in ["WHOSE LINE SAT 8:30P", "HARRY POTTER SUNDAY 10:30P", "HARRY POTTER23", "20/20 SATURDAY-3/24",
          "LAW & ORDER 2/20", "FRONTLINE 4202", "THE DRAGON SUNDAY", "NIGHT COURT SAT 230P",
          "SATURDAY NIGHT LIVE SAT 1130P", "COPS RELOADED TUES10:30PM", "ELLEN 9PM", "POLICE 24/7-MON", "MASTERPIECE 5323", "JEFFERSONS, THE FR 8P", "HIMYM TUE 9P"]:
    print(f"  {t:30} {show(t)}")

Glued units split: 2,492 (every name still rebuilds exactly)

Units seen in 20+ names: 3,720

100 MOST code-like:
SAT 0.92, FRI 0.89, WED 0.89, THU 0.89, TUE 0.88, 4P 0.88, 8P 0.88, 2P 0.88, 6P 0.88, 9P 0.88, SUN 0.88, 7P 0.88, 5P 0.87, 1P 0.87, 12P 0.87, 3P 0.87, 11A 0.87, 10P 0.87, 10A 0.87, SP 0.87, 12A 0.87, 8A 0.86, 6A 0.86, 3A 0.86, 2A 0.86, 11P 0.85, 4A 0.85, 9A 0.85, 1A 0.85, 11 0.84, 7A 0.84, 5A 0.84, 12 0.83, MON 0.82, 3B 0.82, 2ND 0.81, 1B 0.81, 3D 0.81, 330PM 0.80, 630PM 0.80, 955AM 0.80, 15AM 0.80, 9 0.79, 5AM 0.79, 530PM 0.79, 45A 0.79, 7AM 0.78, 245P 0.78, 6 0.78, 00A 0.78, 25 0.78, 50TH 0.77, 104R 0.77, 830AM 0.77, 2B 0.77, 6AM 0.77, 26 0.77, 230PM 0.77, 930AM 0.76, SU 0.76, 23 0.76, 31 0.76, 8AM 0.76, SNEAK 0.76, 28 0.76, 27 0.75, 930A 0.75, 00P 0.75, 130P 0.75, 103R 0.75, DT 0.75, 430P 0.75, 230P 0.74, 4AM 0.74, 830A 0.74, 230A 0.74, 330P 0.74, 15PM 0.74, 13TH 0.74, 630P 0.74, 530P 0.74, 130A 0.74, 9AM 0.74, 102R 0.74, 29 0.74, 101R 0.74, 730P 0.74, 2AM 0.74, 1130A 0.

In [22]:
show('Men into SPACE 330am sun')

'MEN 0.13  INTO 0.03  SPACE 0.07  330AM 0.84  SUN 0.88'

In [36]:
import math
from collections import defaultdict

# ===========================================================================
# STEP 3: relation check (your rule) - does each unit relate to OTHER units of the same name through an IMDb title?
# STEP 4: decision - keep or remove each unit
# ===========================================================================

# ---- settings ----
COVER_MIN     = 0.4    # at least 40% of the title (weighted by word rarity) must be in the name: WHOSE LINE for Whose Line Is It Anyway?
EVIDENCE_MIN  = 6.0    # the tied NAME words must be rare enough together (THE + 13 is not enough; 60 + MINUTES is)
RARE_SEEDS    = 3      # look up candidate titles through the name's 3 rarest words
SEED_MAX_DF   = 2000   # a word in more titles than this is too common to look titles up by
WORD_CUTOFF   = 0.5    # an unrelated letter word is removed only if its code score is at least this

# ---- the IMDb reference: which titles can relate words ----
# Your notebook only loaded imdb_tv.csv, so the full IMDb file is loaded here (only the two columns needed).
IMDB_FULL = "/home/ubuntu/string-matching-algorithm/imb_title_set_updated.csv"
if "imdb" not in globals():
    try:
        imdb = pd.read_csv(IMDB_FULL, usecols=["TitleUS", "TitleType"], encoding="utf-8-sig")
    except UnicodeDecodeError:
        imdb = pd.read_csv(IMDB_FULL, usecols=["TitleUS", "TitleType"], encoding="latin-1")

PROTECT_TYPES = ["tvSeries", "tvMiniSeries", "tvMovie", "tvSpecial", "movie"]
imdb_ref = (imdb.loc[imdb["TitleType"].isin(PROTECT_TYPES), ["TitleUS"]]
            .dropna().drop_duplicates().reset_index(drop=True))
# For a quick test with tvSeries only, use this line instead:  imdb_ref = imdb_tv[["TitleUS"]]
print(f"IMDb titles used for the relation check: {len(imdb_ref):,}")

title_units, title_joins = [], []
unit_index, join_index, df = defaultdict(set), defaultdict(set), Counter()
imdb_chains = Counter()                                         # numbers written together in IMDb titles: 20/20, 9-1-1, 24/7

def is_number_unit(h):
    """Starts with a digit: 2, 4202, 8P, 830P, 1ST. Units that start with letters (H2O, MST3K, EP1) are judged as words."""
    return h[:1] == "9"

def is_word_unit(h):
    return not is_number_unit(h)

def number_chains(units):
    """Numbers written together with no space, as (positions, text): 20/20, 9-1-1, 3/20/20, 1230A/20/20."""
    chains, run = [], []
    for i, (s, u, h) in enumerate(units):
        if is_number_unit(h) and run and run[-1] == i - 1 and is_tight(s):
            run.append(i)
        else:
            if len(run) >= 2:
                chains.append(run)
            run = [i] if is_number_unit(h) else []
    if len(run) >= 2:
        chains.append(run)
    return [(run, "".join((units[k][0] if k != run[0] else "") + units[k][1] for k in run)) for run in chains]

def chain_ok(units, i):
    """A number written together with other numbers is kept only if IMDb writes the WHOLE thing the same way:
    20/20 and 9-1-1 pass; 3/20/20 (a date) and 1230A/20/20 do not, even though they contain 20/20."""
    for run, text in number_chains(units):
        if i in run:
            return imdb_chains[text] >= PAIR_MIN_IMDB
    return True

def joins_with_parts(units):
    """Joins plus the units they are made of: ODD+PARENTS -> ('ODDPARENTS', ('ODD','PARENTS'))."""
    out = []
    for i in range(1, len(units)):
        if not is_clear_break(units[i][0]):
            out.append((units[i-1][1] + units[i][1], (i-1, i)))
    run = [0]
    for i in range(1, len(units) + 1):
        if i < len(units) and is_tight(units[i][0]):
            run.append(i)
        else:
            if len(run) >= 3:
                out.append(("".join(units[k][1] for k in run), tuple(run)))
            run = [i]
    return out

for tid, title in enumerate(imdb_ref["TitleUS"]):
    us, _ = split_units(title)
    words = tuple(u for s, u, h in us)
    title_units.append(words)
    title_joins.append([(j, tuple(words[k] for k in parts)) for j, parts in joins_with_parts(us)])
    for u in set(words):
        unit_index[u].add(tid)
        df[u] += 1
    for j, parts in title_joins[-1]:
        join_index[j].add(tid)
    for run, text in number_chains(us):
        imdb_chains[text] += 1
N_REF = len(imdb_ref)
IDF = {t: math.log(N_REF / n) for t, n in df.items()}
title_sets = [frozenset(words) for words in title_units]
title_joinsets = [frozenset(j for j, parts in tj) for tj in title_joins]
title_total = [sum(IDF[t] for t in ts) for ts in title_sets]
single_titles = {words[0] for words in title_units if len(words) == 1}          # titles that are one unit: 24, FRONTLINE

LOG_N = math.log(N_REF)

letters_cache = {}
def is_letters(u):
    if u not in letters_cache:
        letters_cache[u] = not any(ch.isdigit() for ch in u)
    return letters_cache[u]

def relate(units, codes):
    """Finds EVERY plausible IMDb title: it ties 2 or more real words or numbers of the name together (a code such
    as SUN or SAT never counts as one of the two), is at least 40% present, and shares rare enough name words.
    Returns (title row, positions), best first. Cleaning does not choose between these titles; the later
    matching steps do, with year, show time and the rest."""
    words = [u for s, u, h in units]
    where = defaultdict(set)                                    # string -> positions in the name it stands for
    for i, w in enumerate(words):
        where[w].add(i)
    name_joins = joins_with_parts(units)
    for j, parts in name_joins:                                 # ODD PARENTS can stand for ODDPARENTS
        where[j] |= set(parts)
    keys = set(where)
    # look titles up through the rarest NAME-LIKE words (not SAT or TUES, which are rare in IMDb but are codes)
    seeds = sorted({w for w, c in zip(words, codes) if is_letters(w) and c < WORD_CUTOFF and 0 < df[w] <= SEED_MAX_DF},
                   key=lambda w: df[w])[:RARE_SEEDS]
    candidates = set()
    for w in seeds:                                             # titles holding a seed AND another word of the name
        titles_w = unit_index[w]
        for k in keys:
            if k != w and k in unit_index:
                candidates |= titles_w & unit_index[k]
    for k in keys:                                              # SPIDERMAN can meet SPIDER-MAN, S.W.A.T. can meet SWAT
        if k in join_index and len(join_index[k]) <= SEED_MAX_DF:
            candidates |= join_index[k]
        if k not in words and 0 < df[k] <= SEED_MAX_DF:        # ODD PARENTS can meet ODDPARENTS
            candidates |= unit_index[k]
    passed = []                                                 # every title that ties 2+ words of the name together
    for tid in candidates:
        present = title_sets[tid] & keys
        joined = title_joinsets[tid] & keys
        if len(present) < 2 and not joined:                    # most titles share only the seed word: skip fast
            continue
        present = set(present)
        pos = set()
        for t in present:
            pos |= where[t]
        if joined:
            for j, parts in title_joins[tid]:
                if j in joined:
                    present.update(parts); pos |= where[j]
        anchors = [i for i in pos if is_number_unit(units[i][2]) or codes[i] < WORD_CUTOFF]
        if len(anchors) < 2:                                   # must tie 2+ real words/numbers, not LUCY + SUN or NIGHT + SAT
            continue
        weight = sum(IDF[t] for t in present)
        cover = weight / title_total[tid]
        if cover < COVER_MIN:
            continue
        evidence = sum(IDF.get(words[i], LOG_N) for i in anchors if is_word_unit(units[i][2]))
        if evidence < EVIDENCE_MIN:                            # needs real name words, not 11P + M
            continue
        passed.append((evidence * cover, tid, pos))           # order: real name words covered x how complete the title is
    passed.sort(key=lambda x: -x[0])
    return [(tid, pos) for score, tid, pos in passed]

def block_of(units, i):
    """The unit itself, or the whole run of numbers it is written with (24/7, 9-1-1, 12/30)."""
    for run, text in number_chains(units):
        if i in run:
            return run
    return [i]

def protected(units, codes, block, titles):
    """A number or a code-like word is protected only if a plausible title has the SAME words in the SAME order:
    the block itself plus the real word right next to it. 60 MINUTES: '60 Minutes' has 60 then MINUTES.
    ALL IN THE FMLY SUN: 'All in the Sun' does not count, because SUN follows FMLY in the name, not THE."""
    words = [u for s, u, h in units]
    seq = [words[k] for k in block]
    for k_side, step in ((block[0] - 1, -1), (block[-1] + 1, +1)):
        if not (0 <= k_side < len(units)) or not (is_number_unit(units[k_side][2]) or codes[k_side] < WORD_CUTOFF):
            continue
        if is_number_unit(units[k_side][2]) and is_number_unit(units[block[0]][2]):
            continue                                           # the neighbour must be a real word, not another number
        want = [words[k_side]] + seq if step < 0 else seq + [words[k_side]]
        n = len(want)
        for tid, pos in titles:
            t = title_units[tid]
            if any(list(t[a:a + n]) == want for a in range(len(t) - n + 1)):
                return True
    return False

def decide(units, codes, titles=None):
    """Returns one reason per unit. Kept: related, pair, title, word. Removed: number, code."""
    if titles is None:
        titles = relate(units, codes)
    tied = set().union(*(pos for tid, pos in titles)) if titles else set()
    paired = set()                                             # 20/20, 9-1-1: the whole written chain is an IMDb title
    for run, text in number_chains(units):
        if imdb_chains[text] >= PAIR_MIN_IMDB and any(codes[k] < unit_score(units[k][1], units[k][2]) for k in run):
            paired |= set(run)
    why = []
    for i, ((s, u, h), c) in enumerate(zip(units, codes)):
        if is_number_unit(h) or c >= WORD_CUTOFF:              # numbers and code-like words need a title's protection
            if i in tied and protected(units, codes, block_of(units, i), titles):
                why.append("related")
            elif i in paired:
                why.append("pair")                             # 20/20, 24/7 written alone
            else:
                why.append("number" if is_number_unit(h) else "code")
        else:                                                  # real words stay
            why.append("related" if i in tied else "word")
    if all(w in ("number", "code") for w in why):
        if units and units[0][1] in single_titles and is_number_unit(units[0][2]) and chain_ok(units, 0):
            why[0] = "title"                                   # 24 SUN 9P: the show 24 is the whole program name
        else:
            why = ["empty" for _ in why]                       # never empty a name
    return why

KEEP = {"related", "pair", "title", "word", "empty"}

def processed(units, why):
    out, prev_kept = "", False
    for (s, u, h), w in zip(units, why):
        if w in KEEP:
            out += (s if prev_kept else " ") + u
            prev_kept = True
        else:
            prev_kept = False
    return out.strip(" ").strip("".join(set("-/:;,.!?*'& ")))

MAX_TITLES_SHOWN = 10                                          # how many of the passing titles to keep in the output column
all_titles = [relate(us, cs) for us, cs in zip(raw_names["units"], raw_names["code"])]
raw_names["why"] = [decide(us, cs, t) for us, cs, t in zip(raw_names["units"], raw_names["code"], all_titles)]
raw_names["imdb_titles"] = [[imdb_ref["TitleUS"].iloc[tid] for tid, pos in t[:MAX_TITLES_SHOWN]] for t in all_titles]
raw_names["processed_name"] = [processed(us, w) for us, w in zip(raw_names["units"], raw_names["why"])]
raw_names["removed_units"] = [[u for (s, u, h), x in zip(us, w) if x not in KEEP] for us, w in zip(raw_names["units"], raw_names["why"])]
raw_names["no_program_name"] = raw_names["why"].map(lambda w: "empty" in w)
raw_names["maybe_cut"] = raw_names["nielsen_program_name"].str.len() >= 25

# ---- checks ----
reasons = Counter(x for w in raw_names["why"] for x in w)
print("Units by reason:", dict(reasons.most_common()))
print(f"Names with nothing left to keep (flagged no_program_name): {raw_names['no_program_name'].sum():,}")
n_titles = [len(t) for t in all_titles]
print(f"IMDb titles tying words together, per name: none {sum(n == 0 for n in n_titles):,}, one {sum(n == 1 for n in n_titles):,}, "
      f"2-10 {sum(2 <= n <= 10 for n in n_titles):,}, more than 10 {sum(n > 10 for n in n_titles):,}")

display(raw_names.sample(25, random_state=7)[["nielsen_program_name", "processed_name", "removed_units"]])

IMDb titles used for the relation check: 1,166,847
Units by reason: {'related': 337121, 'number': 164935, 'word': 144331, 'code': 72600, 'pair': 304, 'title': 105, 'empty': 52}
Names with nothing left to keep (flagged no_program_name): 28
IMDb titles tying words together, per name: none 49,556, one 22,208, 2-10 47,007, more than 10 55,250


,nielsen_program_name,processed_name,removed_units
27825,BIRDGIRL,BIRDGIRL,[]
21917,BUNDESLIGA ESPECIAL - R,BUNDESLIGA ESPECIAL,[R]
67051,"END OF THE AFFAIR, THE","END OF THE AFFAIR, THE",[]
64560,"VECINA, LA DT","VECINA, LA",[DT]
158674,WHO IS VLADIMIR PUTIN?,WHO IS VLADIMIR PUTIN,[]
84546,JEFFERSONS THKSGVG 6A,JEFFERSONS THKSGVG,[6A]
52566,UEFA WMN EURO TDAY-2P-7/5,UEFA WMN EURO TDAY,"[2P, 7, 5]"
107612,LIFE AND PASSION OF CHRIS,LIFE AND PASSION OF CHRIS,[]
76803,LAUGH LIST SUN 1230P,LAUGH LIST,"[SUN, 1230P]"
159799,TWO STEPS HOME SNEAK,TWO STEPS HOME,[SNEAK]


In [37]:
EXPLAIN = {
    "related": "kept: IMDb ties it to another word of this name",
    "pair":    "kept: written pair that is a real title (20/20, 24/7)",
    "title":   "kept: a number that is a whole IMDb title, at the start",
    "word":    "kept: a real word",
    "empty":   "kept: everything looked like schedule wording, so nothing was removed",
    "number":  "REMOVED: a number IMDb does not tie to this name",
    "code":    "REMOVED: schedule wording (follows many different programs)",
}

def test_name(*names, show_titles=5):
    for name in names:
        units = split_glued(split_units(name)[0])   # same splitting your pipeline uses for raw_names["units"]
        codes = code_scores(units)
        titles = relate(units, codes)                # list of (tid, positions), best first
        why = decide(units, codes, titles)
        print(f"\n{name}")
        print(f"  processed name: {processed(units, why)!r}")
        if titles:
            print(f"  IMDb titles tying words ({len(titles)} total, top {show_titles}):")
            for tid, pos in titles[:show_titles]:
                tied = [units[i][1] for i in sorted(pos)]
                print(f"    {imdb_ref['TitleUS'].iloc[tid]!r:45} ties {tied}")
        else:
            print("  IMDb titles: (none ties two words together)")
        print(f"  {'unit':14} {'shape':8} {'code':>5}  decision")
        for (s, u, h), c, w in zip(units, codes, why):
            print(f"  {u:14} {h:8} {c:5.2f}  {EXPLAIN[w]}")

test_name("90IN60COPALIBERTADORESF-R","special")


90IN60COPALIBERTADORESF-R
  processed name: '90IN60COPALIBERTADORESF-R'
  IMDb titles: (none ties two words together)
  unit           shape     code  decision
  90IN60COPALIBERTADORESF 99A99A    0.01  kept: everything looked like schedule wording, so nothing was removed
  R              A         0.72  kept: everything looked like schedule wording, so nothing was removed

special
  processed name: 'SPECIAL'
  IMDb titles: (none ties two words together)
  unit           shape     code  decision
  SPECIAL        A         0.52  kept: everything looked like schedule wording, so nothing was removed


In [43]:
# ===========================================================================
# STEP 5: CLEAN THE 10,000 SYNTHETIC NAMES AND MEASURE HOW WELL IT WORKED
# Needs cells 1 to 5 to have run first (it uses split_units, split_glued, code_scores, relate, decide, processed).
# ===========================================================================
import time
from collections import Counter

SYNTH_CSV = "/home/ubuntu/string-matching-algorithm/synthetic_nielsen_10000.csv"   # where you put the file
syn = pd.read_csv(SYNTH_CSV, keep_default_na=False, dtype=str)

def unit_truth(units, mask):
    """keep / remove / mixed for every unit, read from the character labels: K = title, R = noise, . = separator."""
    out, pos = [], 0
    for sep, u, h in units:
        pos += len(sep)
        labels = set(mask[pos:pos + len(u)]) - {"."}
        out.append("keep" if labels == {"K"} else "remove" if labels == {"R"} else "mixed")
        pos += len(u)
    return out

rows, t0 = [], time.time()
for k, r in enumerate(syn.itertuples(index=False)):
    name, mask = r.nielsen_program_name, r.label_mask
    assert normalize(name) == name, f"row {r.id}: name is not in normalized form"
    units  = split_glued(split_units(name)[0])
    codes  = code_scores(units)
    titles = relate(units, codes)
    why    = decide(units, codes, titles)
    truth  = unit_truth(units, mask)
    kept   = [w in KEEP for w in why]
    pairs  = list(zip([u for s, u, h in units], truth, kept))
    rows.append({
        "processed_name":   processed(units, why),
        "n_title_units":    sum(t == "keep" for u, t, k in pairs),
        "n_title_kept":     sum(t == "keep" and k for u, t, k in pairs),
        "n_noise_units":    sum(t == "remove" for u, t, k in pairs),
        "n_noise_removed":  sum(t == "remove" and not k for u, t, k in pairs),
        "lost_title_units": [u for u, t, k in pairs if t == "keep" and not k],      # worst error: a title word was removed
        "noise_left":       [u for u, t, k in pairs if t == "remove" and k],        # noise that stayed in the name
        "mixed_units":      [u for u, t, k in pairs if t == "mixed"],               # title and noise glued, not split
        "imdb_titles_top3": [imdb_ref["TitleUS"].iloc[tid] for tid, pos in titles[:3]],
        "units_and_reasons": " ".join(f"{u}:{w}" for (s, u, h), w in zip(units, why)),
    })
    if (k + 1) % 2000 == 0:
        print(f"  cleaned {k + 1:,} names ({time.time() - t0:.0f}s)")

res = pd.concat([syn, pd.DataFrame(rows)], axis=1)
res["title_word_lost"] = res["lost_title_units"].map(len) > 0
res["noise_left_over"] = res["noise_left"].map(len) > 0
res["perfect"]         = ~res["title_word_lost"] & ~res["noise_left_over"] & (res["mixed_units"].map(len) == 0)
res["same_as_expected"] = res["processed_name"] == res["expected_clean"]

def summary(g):
    removed_total = (g["n_title_units"] - g["n_title_kept"]).sum() + g["n_noise_removed"].sum()
    return pd.Series({
        "names":                   len(g),
        "title words kept %":      100 * g["n_title_kept"].sum() / max(g["n_title_units"].sum(), 1),
        "noise removed %":         100 * g["n_noise_removed"].sum() / g["n_noise_units"].sum() if g["n_noise_units"].sum() else float("nan"),
        "removed that was noise %": 100 * g["n_noise_removed"].sum() / removed_total if removed_total else float("nan"),
        "names perfect %":         100 * g["perfect"].mean(),
        "names losing a title word %": 100 * g["title_word_lost"].mean(),
        "names with noise left %": 100 * g["noise_left_over"].mean(),
        "output = expected %":     100 * g["same_as_expected"].mean(),
    })

pd.set_option("display.float_format", "{:.1f}".format)
pd.set_option("display.width", 250)
overall = summary(res)
print(f"\nCleaned {len(res):,} names in {time.time() - t0:.0f}s\n")
print("OVERALL")
print(overall.to_string())
print(f"\nFor comparison, removing nothing at all: title words kept 100%, noise removed 0%, "
      f"names perfect {100 * (res['n_noise_units'] == 0).mean():.1f}%")

by_cat = pd.DataFrame({cat: summary(g) for cat, g in res.groupby("category")}).T.sort_values("names perfect %")
print("\nBY CATEGORY (worst first)")
display(by_cat)

print("\nTitle words removed most often (should have been kept):")
print(Counter(u for us in res["lost_title_units"] for u in us).most_common(30))
print("\nNoise left in most often (should have been removed):")
print(Counter(u for us in res["noise_left"] for u in us).most_common(30))

failures = res[~res["perfect"]]
cols = ["category", "nielsen_program_name", "expected_clean", "processed_name", "lost_title_units", "noise_left",
        "mixed_units", "imdb_title", "imdb_titles_top3", "units_and_reasons"]
failures[cols].to_csv("synthetic_failures.csv", index=False)
res.drop(columns=["label_mask"]).to_csv("synthetic_results.csv", index=False)
print(f"\nSaved {len(failures):,} failures to synthetic_failures.csv and every row to synthetic_results.csv")

print("\nTwo failures per category:")
display(failures.groupby("category").head(2).sort_values("category")[["category", "nielsen_program_name", "expected_clean", "processed_name"]])

  cleaned 2,000 names (1s)
  cleaned 4,000 names (3s)
  cleaned 6,000 names (4s)
  cleaned 8,000 names (6s)
  cleaned 10,000 names (7s)

Cleaned 10,000 names in 7s

OVERALL
names                         10000.0
title words kept %               99.0
noise removed %                  86.6
removed that was noise %         98.7
names perfect %                  76.0
names losing a title word %       1.9
names with noise left %          22.0
output = expected %              76.0

For comparison, removing nothing at all: title words kept 100%, noise removed 0%, names perfect 11.6%

BY CATEGORY (worst first)


,names,title words kept %,noise removed %,removed that was noise %,names perfect %,names losing a title word %,names with noise left %,output = expected %
full_day_noise,333.0,100.0,43.0,100.0,0.0,0.0,100.0,0.0
network_noise,332.0,99.9,41.3,99.6,14.2,0.3,85.8,14.2
prefix_noise,334.0,99.6,20.4,95.8,20.4,0.6,79.6,20.4
ambiguous_word_noise,333.0,99.5,68.9,99.3,42.0,0.9,57.7,42.0
noise_heavy,333.0,96.8,88.4,99.1,54.4,3.9,44.4,54.4
codes,334.0,100.0,74.8,100.0,64.7,0.0,35.3,64.7
episode_number,334.0,99.7,71.4,99.4,69.8,0.6,29.6,69.8
number_title_number_noise,333.0,94.0,90.6,92.6,74.2,13.2,13.2,74.2
encoding_question_mark,334.0,94.0,96.5,92.3,76.9,17.1,6.9,76.9
spanish,333.0,99.9,86.4,99.8,77.8,0.3,21.9,77.8



Title words removed most often (should have been kept):
[('M', 31), ('1', 12), ('R', 12), ('F', 12), ('10', 8), ('SPECIAL', 8), ('2', 7), ('W', 6), ('3', 6), ('30', 5), ('TH', 4), ('MON', 4), ('POST', 4), ('11', 4), ('FR', 4), ('5', 4), ('24', 3), ('18', 3), ('8', 3), ('7', 3), ('20', 2), ('36', 2), ('50', 2), ('SUN', 2), ('60', 2), ('1809', 2), ('TV3', 2), ('101', 2), ('QT6', 2), ('5MINUTENSHOW', 2)]

Noise left in most often (should have been removed):
[('MOVIE', 172), ('ENCORE', 127), ('ET', 94), ('AM', 90), ('PM', 87), ('PREM', 86), ('RPT', 81), ('SSN', 63), ('NBC', 62), ('WE', 62), ('PART', 62), ('TUES', 58), ('PREMIERE', 57), ('TLMD', 56), ('MONDAY', 55), ('SATURDAY', 55), ('ABC', 54), ('ON', 53), ('EM', 53), ('CBS', 50), ('TU', 50), ('SUNDAY', 49), ('MO', 49), ('FOX', 49), ('THURSDAY', 48), ('THR', 47), ('L', 47), ('1', 47), ('REPLAY', 46), ('FRIDAY', 45)]

Saved 2,395 failures to synthetic_failures.csv and every row to synthetic_results.csv

Two failures per category:


,category,nielsen_program_name,expected_clean,processed_name
322,abbreviated,ALTRNT ROUTE SAT 7 AM,ALTRNT ROUTE,ALTRNT ROUTE AM
217,abbreviated,SLVG CITY WE-9PM,SLVG CITY,SLVG CITY WE
82,ambiguous_word_noise,CALIFORNIA PREMIERE 10PM,CALIFORNIA,CALIFORNIA PREMIERE
116,ambiguous_word_noise,INUYASHA MOVIE,INUYASHA,INUYASHA MOVIE
15,article_dropped,BIG CHEESE SUN 2 AM,BIG CHEESE,BIG CHEESE 2 AM
925,article_dropped,COWBOYS TU 12AM,COWBOYS,COWBOYS TU
612,article_moved,"HOSPITAL, THE SAT 6 AM","HOSPITAL, THE","HOSPITAL, THE 6 AM"
748,article_moved,"BOUNCE,THE WE 3:00AM","BOUNCE,THE","BOUNCE,THE WE"
5098,clean_title_only,10 000 KRONERS-DUELLEN,10 000 KRONERS-DUELLEN,000 KRONERS-DUELLEN
1063,code_word_title,LIVE FLOOR SHOW MO 6PM,LIVE FLOOR SHOW,LIVE FLOOR SHOW MO


#####earlier step 4(one imdb title matcher)

In [44]:
import math
from collections import defaultdict

# ===========================================================================
# STEP 3: relation check (your rule) - does each unit relate to OTHER units of the same name through an IMDb title?
# STEP 4: decision - keep or remove each unit
# ===========================================================================

# ---- settings ----
COVER_MIN     = 0.4    # at least 40% of the title (weighted by word rarity) must be in the name: WHOSE LINE for Whose Line Is It Anyway?
EVIDENCE_MIN  = 6.0    # the shared LETTER words must be rare enough together (THE + 13 is not enough; 60 + MINUTES is)
RARE_SEEDS    = 3      # look up candidate titles through the name's 3 rarest words
SEED_MAX_DF   = 2000   # a word in more titles than this is too common to look titles up by
WORD_CUTOFF   = 0.5    # an unrelated letter word is removed only if its code score is at least this

# ---- the IMDb reference: which titles can relate words ----
# Your notebook only loaded imdb_tv.csv, so the full IMDb file is loaded here (only the two columns needed).
IMDB_FULL = "/home/ubuntu/string-matching-algorithm/imb_title_set_updated.csv"
if "imdb" not in globals():
    try:
        imdb = pd.read_csv(IMDB_FULL, usecols=["TitleUS", "TitleType"], encoding="utf-8-sig")
    except UnicodeDecodeError:
        imdb = pd.read_csv(IMDB_FULL, usecols=["TitleUS", "TitleType"], encoding="latin-1")

# PROTECT_TYPES = ["tvSeries", "tvMiniSeries", "tvMovie", "tvSpecial", "movie"]
PROTECT_TYPES = ["tvSeries"]
imdb_ref = (imdb.loc[imdb["TitleType"].isin(PROTECT_TYPES), ["TitleUS"]]
            .dropna().drop_duplicates().reset_index(drop=True))
# For a quick test with tvSeries only, use this line instead:  imdb_ref = imdb_tv[["TitleUS"]]
print(f"IMDb titles used for the relation check: {len(imdb_ref):,}")

title_units, title_joins = [], []
unit_index, join_index, df = defaultdict(set), defaultdict(set), Counter()
imdb_chains = Counter()                                         # numbers written together in IMDb titles: 20/20, 9-1-1, 24/7

def is_number_unit(h):
    """Starts with a digit: 2, 4202, 8P, 830P, 1ST. Units that start with letters (H2O, MST3K, EP1) are judged as words."""
    return h[:1] == "9"

def is_word_unit(h):
    return not is_number_unit(h)

def number_chains(units):
    """Numbers written together with no space, as (positions, text): 20/20, 9-1-1, 3/20/20, 1230A/20/20."""
    chains, run = [], []
    for i, (s, u, h) in enumerate(units):
        if is_number_unit(h) and run and run[-1] == i - 1 and is_tight(s):
            run.append(i)
        else:
            if len(run) >= 2:
                chains.append(run)
            run = [i] if is_number_unit(h) else []
    if len(run) >= 2:
        chains.append(run)
    return [(run, "".join((units[k][0] if k != run[0] else "") + units[k][1] for k in run)) for run in chains]

def chain_ok(units, i):
    """A number written together with other numbers is kept only if IMDb writes the WHOLE thing the same way:
    20/20 and 9-1-1 pass; 3/20/20 (a date) and 1230A/20/20 do not, even though they contain 20/20."""
    for run, text in number_chains(units):
        if i in run:
            return imdb_chains[text] >= PAIR_MIN_IMDB
    return True

def joins_with_parts(units):
    """Joins plus the units they are made of: ODD+PARENTS -> ('ODDPARENTS', ('ODD','PARENTS'))."""
    out = []
    for i in range(1, len(units)):
        if not is_clear_break(units[i][0]):
            out.append((units[i-1][1] + units[i][1], (i-1, i)))
    run = [0]
    for i in range(1, len(units) + 1):
        if i < len(units) and is_tight(units[i][0]):
            run.append(i)
        else:
            if len(run) >= 3:
                out.append(("".join(units[k][1] for k in run), tuple(run)))
            run = [i]
    return out

for tid, title in enumerate(imdb_ref["TitleUS"]):
    us, _ = split_units(title)
    words = tuple(u for s, u, h in us)
    title_units.append(words)
    title_joins.append([(j, tuple(words[k] for k in parts)) for j, parts in joins_with_parts(us)])
    for u in set(words):
        unit_index[u].add(tid)
        df[u] += 1
    for j, parts in title_joins[-1]:
        join_index[j].add(tid)
    for run, text in number_chains(us):
        imdb_chains[text] += 1
N_REF = len(imdb_ref)
IDF = {t: math.log(N_REF / n) for t, n in df.items()}
title_sets = [frozenset(words) for words in title_units]
title_joinsets = [frozenset(j for j, parts in tj) for tj in title_joins]
title_total = [sum(IDF[t] for t in ts) for ts in title_sets]
single_titles = {words[0] for words in title_units if len(words) == 1}          # titles that are one unit: 24, FRONTLINE

LOG_N = math.log(N_REF)

letters_cache = {}
def is_letters(u):
    if u not in letters_cache:
        letters_cache[u] = not any(ch.isdigit() for ch in u)
    return letters_cache[u]

def relate(units, codes):
    """Finds the IMDb title that best explains the name. Returns the positions of the name units that
    relate to each other through it, and that title's row number (empty, None if no title ties 2+ units together)."""
    words = [u for s, u, h in units]
    where = defaultdict(set)                                    # string -> positions in the name it stands for
    for i, w in enumerate(words):
        where[w].add(i)
    name_joins = joins_with_parts(units)
    for j, parts in name_joins:                                 # ODD PARENTS can stand for ODDPARENTS
        where[j] |= set(parts)
    keys = set(where)
    # look titles up through the rarest NAME-LIKE words (not SAT or TUES, which are rare in IMDb but are codes)
    seeds = sorted({w for w, c in zip(words, codes) if is_letters(w) and c < WORD_CUTOFF and 0 < df[w] <= SEED_MAX_DF},
                   key=lambda w: df[w])[:RARE_SEEDS]
    candidates = set()
    for w in seeds:                                             # titles holding a seed AND another word of the name
        titles_w = unit_index[w]
        for k in keys:
            if k != w and k in unit_index:
                candidates |= titles_w & unit_index[k]
    for k in keys:                                              # SPIDERMAN can meet SPIDER-MAN, S.W.A.T. can meet SWAT
        if k in join_index and len(join_index[k]) <= SEED_MAX_DF:
            candidates |= join_index[k]
        if k not in words and 0 < df[k] <= SEED_MAX_DF:        # ODD PARENTS can meet ODDPARENTS
            candidates |= unit_index[k]
    best, best_tid, best_score = set(), None, 0.0
    for tid in candidates:
        present = title_sets[tid] & keys
        joined = title_joinsets[tid] & keys
        if len(present) < 2 and not joined:                    # most titles share only the seed word: skip fast
            continue
        present = set(present)
        pos = set()
        for t in present:
            pos |= where[t]
        if joined:
            for j, parts in title_joins[tid]:
                if j in joined:
                    present.update(parts); pos |= where[j]
        if len(pos) < 2:                                       # a word must relate to ANOTHER word of the name
            continue
        weight = sum(IDF[t] for t in present)
        cover = weight / title_total[tid]
        if cover < COVER_MIN:
            continue
        evidence = sum(IDF.get(words[i], LOG_N) for i in pos if is_word_unit(units[i][2]) and codes[i] < WORD_CUTOFF)
        if evidence < EVIDENCE_MIN:                            # needs real name words, not 11P + M
            continue
        score = weight * cover
        if score > best_score:
            best, best_tid, best_score = pos, tid, score
    return best, best_tid

def number_ok(units, i, related):
    """A number stays related only if it sits right next to another related unit of the name (60 MINUTES,
    2 BROKE GIRLS) and is not part of a date or time (12 in 12/30) unless IMDb writes it that way (24/7)."""
    return chain_ok(units, i) and ((i - 1 in related) or (i + 1 in related))

def decide(units, codes):
    """Returns one reason per unit. Kept: related, pair, title, word. Removed: number, code."""
    related, tid = relate(units, codes)
    for _ in range(2):                                         # numbers must earn their place next to the title's words
        related = {i for i in related if is_word_unit(units[i][2]) or number_ok(units, i, related)}
    paired = set()                                             # 20/20, 9-1-1: the whole written chain is an IMDb title
    for run, text in number_chains(units):
        if imdb_chains[text] >= PAIR_MIN_IMDB and any(codes[k] < unit_score(units[k][1], units[k][2]) for k in run):
            paired |= set(run)
    why = []
    for i, ((s, u, h), c) in enumerate(zip(units, codes)):
        if i in related:
            why.append("related")                              # IMDb ties it to another word of the name
        elif is_number_unit(h):
            why.append("pair" if i in paired else "number")    # 20/20, 24/7 stay; times, dates, episode numbers go
        else:
            why.append("word" if c < WORD_CUTOFF else "code")  # THE DRAGON SUNDAY keeps SUNDAY; SAT goes
    if all(w in ("number", "code") for w in why):
        u0 = units[0][1] if units else ""
        if units and u0 in single_titles and is_number_unit(units[0][2]) and number_ok(units, 0, {1}):
            why[0] = "title"                                   # 24 SUN 9P: the show 24 is the whole program name
        else:
            why = ["empty" for _ in why]                       # never empty a name
    return why

KEEP = {"related", "pair", "title", "word", "empty"}

def processed(units, why):
    out, prev_kept = "", False
    for (s, u, h), w in zip(units, why):
        if w in KEEP:
            out += (s if prev_kept else " ") + u
            prev_kept = True
        else:
            prev_kept = False
    return out.strip(" ").strip("".join(set("-/:;,.!?*'& ")))

raw_names["why"] = [decide(us, cs) for us, cs in zip(raw_names["units"], raw_names["code"])]
raw_names["processed_name"] = [processed(us, w) for us, w in zip(raw_names["units"], raw_names["why"])]
raw_names["removed_units"] = [[u for (s, u, h), x in zip(us, w) if x not in KEEP] for us, w in zip(raw_names["units"], raw_names["why"])]
raw_names["no_program_name"] = raw_names["why"].map(lambda w: "empty" in w)
raw_names["maybe_cut"] = raw_names["nielsen_program_name"].str.len() >= 25

# ---- checks ----
reasons = Counter(x for w in raw_names["why"] for x in w)
print("Units by reason:", dict(reasons.most_common()))
print(f"Names with nothing left to keep (flagged no_program_name): {raw_names['no_program_name'].sum():,}")

display(raw_names.sample(25, random_state=7)[["nielsen_program_name", "processed_name", "removed_units"]])

IMDb titles used for the relation check: 284,555


KeyboardInterrupt: 

In [19]:
# ===========================================================================
# TEST ANY NIELSEN NAME: test_name("WHOSE LINE SAT 8:30P")   or several: test_name("A", "B")
# ===========================================================================
EXPLAIN = {
    "related": "kept: IMDb ties it to another word of this name",
    "pair":    "kept: written pair that is a real title (20/20, 24/7)",
    "title":   "kept: a number that is a whole IMDb title, at the start",
    "word":    "kept: a word that does not behave like schedule wording",
    "empty":   "kept: everything looked like schedule wording, so nothing was removed",
    "number":  "REMOVED: a number IMDb does not tie to this name",
    "code":    "REMOVED: schedule wording (follows many different programs)",
}

def test_name(*names):
    for name in names:
        units = split_glued(split_units(name)[0])
        codes = code_scores(units)
        related, tid = relate(units, codes)
        why = decide(units, codes)
        print(f"\n{name}")
        print(f"  processed name: {processed(units, why)!r}")
        print(f"  IMDb title used: {imdb_ref['TitleUS'].iloc[tid] if tid is not None else '(none ties two words together)'}")
        print(f"  {'unit':12} {'shape':7} {'code':>5}  decision")
        for (s, u, h), c, w in zip(units, codes, why):
            print(f"  {u:12} {h:7} {c:5.2f}  {EXPLAIN[w]}")

test_name("WHOSE LINE SAT 8:30P", "FRONTLINE 4202", "THE DRAGON SUNDAY")

KeyError: 'WHOSE'

In [55]:
test_name('20/20 4/14/2002 sunday')


20/20 4/14/2002 sunday
  processed name: '20/20 SUNDAY'
  IMDb title used: (none ties two words together)
  unit         shape    code  decision
  20           99       0.03  kept: written pair that is a real title (20/20, 24/7)
  20           99       0.03  kept: written pair that is a real title (20/20, 24/7)
  4            9        0.71  REMOVED: a number IMDb does not tie to this name
  14           99       0.73  REMOVED: a number IMDb does not tie to this name
  2002         9999     0.41  REMOVED: a number IMDb does not tie to this name
  SUNDAY       A        0.19  kept: a word that does not behave like schedule wording
